<a href="https://colab.research.google.com/github/nisaerdemli/imageprocessing/blob/main/%C3%B6dev4_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# 1. Tuğla görselini gri tonlamalı yükle
image_path = 'duvar.jpeg'
image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

if image is None:
    # Eğer duvar.jpeg yoksa bilgisayardan seçip yükleme penceresi aç
    from google.colab import files
    print("Lütfen duvar görselini seçin:")
    yuklenenler = files.upload()
    image_path = list(yuklenenler.keys())[0]
    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

# -------------------------------------------------------------------------
# A) 7x7 Laplacian of Gaussian (LoG - Meksika Şapkası) Çekirdeği
# -------------------------------------------------------------------------
def create_log_kernel(size=7, sigma=1.0):
    k = size // 2
    y, x = np.ogrid[-k:k+1, -k:k+1]
    r2 = x**2 + y**2
    sigma2 = sigma**2
    # LoG formülü
    kernel = -(1.0 / (np.pi * sigma2**2)) * (1.0 - r2 / (2.0 * sigma2)) * np.exp(-r2 / (2.0 * sigma2))
    # DC dengesi (ortalama değeri 0 yapmak için)
    kernel = kernel - np.mean(kernel)
    return kernel.astype(np.float32)

log_7x7 = create_log_kernel(size=7, sigma=1.0)
edges_log = cv2.filter2D(image, cv2.CV_64F, log_7x7)
abs_log = np.uint8(np.clip(np.absolute(edges_log), 0, 255))

# -------------------------------------------------------------------------
# B) Kıyaslama Filtreleri:
# -------------------------------------------------------------------------
# 1. Gauss UYGULANMAMIŞ Ham 3x3 Laplacian (Gürültüye doğrudan maruz kalan)
laplacian_kernel = np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]], dtype=np.float32)
edges_raw_laplacian = cv2.filter2D(image, cv2.CV_64F, laplacian_kernel)
abs_raw_laplacian = np.uint8(np.clip(np.absolute(edges_raw_laplacian), 0, 255))

# 2. Gauss UYGULANMIŞ 3x3 Sobel (Bileşke Kenar)
blurred = cv2.GaussianBlur(image, (3, 3), sigmaX=1.0)
sx = cv2.filter2D(blurred, cv2.CV_64F, np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=np.float32))
sy = cv2.filter2D(blurred, cv2.CV_64F, np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=np.float32))
abs_sobel_mag = np.uint8(np.clip(np.sqrt(sx**2 + sy**2), 0, 255))

# -------------------------------------------------------------------------
# C) Yan Yana Görselleştirme
# -------------------------------------------------------------------------
plt.figure(figsize=(18, 5))

plt.subplot(1, 4, 1)
plt.title("Orijinal Gri Görsel", fontsize=11)
plt.imshow(image, cmap='gray')
plt.axis('off')

plt.subplot(1, 4, 2)
plt.title("Gauss'suz Ham Laplacian\n(Aşırı Gürültü ve Noktacık)", fontsize=11)
plt.imshow(abs_raw_laplacian, cmap='gray')
plt.axis('off')

plt.subplot(1, 4, 3)
plt.title("Gauss + 3x3 Sobel (Bileşke)\n(Belirgin Harç Çizgileri)", fontsize=11)
plt.imshow(abs_sobel_mag, cmap='gray')
plt.axis('off')

plt.subplot(1, 4, 4)
plt.title("7x7 LoG (Meksika Şapkası)\n(Entegre Yumuşatmalı 2. Türev)", fontsize=11)
plt.imshow(abs_log, cmap='gray')
plt.axis('off')

plt.tight_layout()
plt.show()

Lütfen duvar görselini seçin:
